# Run a task and inspect results

Run `qdash config init --project-id your-project-id` in a terminal, then fill in `api_token` in the generated file. Use `--base-url` for a remote deployment. If the file already exists, edit it instead. `QDash()` reads its `default_profile`; use `QDash.from_profile("local")` to select a profile explicitly. See the [Python reference](../../../docs/reference/clients/python.md#connect-to-a-deployment) for setup. Replace the chip ID and choose settings suitable for your hardware.

In [ ]:
from qdash.sdk import QDash
from qdash.sdk.calibration import sweep

qdash = QDash()
CHIP_ID = "YOUR_CHIP_ID"
QID = "0"

## Load Parameters

Review and edit the loaded values. Fill missing inputs before `to_call()`; forbidden inputs require calibration first.

In [ ]:
params = qdash.tasks.load_parameters("CheckT1", chip_id=CHIP_ID, qid=QID)
print("Input:", params.input)
print("Run:", params.run)
print("Missing:", params.missing)

params.run["time_range"] = sweep.linspace(300, 500_000, 51)
call = params.to_call()

## Run Task

Running this cell starts a measurement; re-running it starts another. This call records results without updating current calibration values. A wait timeout does not cancel it.

In [ ]:
run = qdash.tasks.run(call, chip_id=CHIP_ID, qid=QID)
execution_id = run.execution_id
print("Execution:", execution_id)
print(run.wait(timeout_seconds=600).status)

## Inspect Results

To inspect an earlier run, set `execution_id` to its ID.

In [ ]:
for execution in qdash.run(execution_id).executions():
    print(execution.execution_id, execution.status)
    for task in execution.tasks:
        print(task.task_id, task.name, task.qid, task.status)
        print("Input:", task.input_parameters)
        print("Run:", task.run_parameters)
        print("Output:", task.output_parameters)

## Show Figures

Paste a task ID from the results above. Display needs IPython. Use `figure.save("result.png")` to save a figure.

In [ ]:
task = qdash.task_result("YOUR_TASK_ID")
for figure in task.figures:
    figure.show()